# 02 — Apply locked parameters to the whole series

Takes the detection parameters locked by `01_calibrate.ipynb` and runs them across every
slice, then produces the per-slice QC summary, the classification k-sweep, the
region-of-interest CSV and the Fos-centric readout.

Same cost split as `01`: the QuPath batch is **EXPENSIVE** (~30 min/slice) and defaults to
a dry run; everything downstream is **CHEAP** pandas over the exported tables and is where
the actual analysis happens. Skip section 2 entirely if the exports already exist.

Run in the `braian` env.

## PARAMS — the only cell you edit

In [1]:
PARAMS = {
    "project": "/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026",

    # Safety. True = the batch cell only PRINTS its commands.
    "dry_run": True,

    # True when results/ is already populated — skips the expensive batch entirely.
    "skip_detection": True,

    # Robust-cut multipliers to sweep. The readout is reported at `k_report`.
    "k_values": [2.0, 2.5, 3.0],
    "k_report": 3.0,

    # Regions for the readout. None -> <project>/regions_of_interest.txt, else these.
    "regions": None,

    # Reactivation conditional: P(target+ | condition+). None -> first two markers
    # declared in pipeline.yml (Fos | TdT for this project).
    "target_marker": None,
    "condition_marker": None,

    # Gate threshold overrides, e.g. {"area_peak_min": 25.0}.
    "thresholds": {},

    # Where the tidy outputs land (relative to <project>/results).
    "readout_csv": "cockpit_readout.csv",
    "regions_csv": "region_of_interest_readout.csv",
    "qc_csv": "cockpit_qc_summary.csv",
}

import sys
from dataclasses import replace
from pathlib import Path

sys.path.insert(0, str(Path("/home/jflab/Analysis/scripts")))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cockpit_checks as cc
import cockpit_regions as creg

PROJECT = Path(PARAMS["project"])
RESULTS = PROJECT / "results"
TH = replace(cc.DEFAULT_THRESHOLDS, k_values=tuple(PARAMS["k_values"]), **PARAMS["thresholds"])
print(f"project : {PROJECT}\ndry_run : {PARAMS['dry_run']}   skip_detection: {PARAMS['skip_detection']}")

project : /home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026
dry_run : True   skip_detection: True


## 1. Config panel — the locked parameters being applied

In [2]:
config = cc.load_config(PROJECT)
anchor_ch = cc.anchor_channel(PROJECT)
MARKERS = config.marker_names

print("pipeline.yml")
print(f"  anchor   : {config.anchor_name}  (channel {anchor_ch})")
for m in config.markers:
    print(f"  marker   : {m['name']:<6} channel {m['channel']:<12} compartment {m['compartment']}")
print(f"  Double+  : {'yes' if config.emit_double else 'no (single marker — no Double+ anywhere)'}")
print(f"  exclude  : {sorted(config.exclude_acronyms) or '(none)'}")
print(f"  k_robust : {cc.k_robust(PROJECT):g}   (sweeping {PARAMS['k_values']})")

print("\nBraiAn.yml — locked detection params")
for k, v in cc.detection_params(PROJECT).items():
    print(f"  {k:<34} {v}")

slices = cc.find_slices(PROJECT)
print(f"\n{len(slices)} slice(s): {', '.join(s.label for s in slices)}")

pipeline.yml
  anchor   : DAPI  (channel DAPI-T4)
  marker   : Fos    channel AF488-T3     compartment nuclear
  marker   : TdT    channel AF568-T2     compartment whole-cell
  Double+  : yes
  exclude  : ['DG-sg', 'VS']
  k_robust : 3   (sweeping [2.0, 2.5, 3.0])

BraiAn.yml — locked detection params
  channel                            DAPI-T4
  classForDetections                 allen_mouse_10um_java
  requestedPixelSizeMicrons          0.6905355
  sigmaMicrons                       2.0
  minAreaMicrons                     20.0
  maxAreaMicrons                     250.0
  cellExpansionMicrons               5.0
  backgroundRadiusMicrons            10
  medianRadiusMicrons                0.0
  watershedPostProcess               True
  histogramThreshold.resolutionLevel 0
  histogramThreshold.smoothWindowSize 15
  histogramThreshold.peakProminence  500
  histogramThreshold.nPeak           2

5 slice(s): wBA1-3_s1, wBA1-3_s2, wBA1-3_s3, wBA1-3_s4, wBA1-3_s5


## 2. Batch detect → classify → export &nbsp;<sub>EXPENSIVE — ~30 min × slices</sub>

Each stage runs project-wide (no `--image`), so QuPath applies it to every entry.
`--save` persists detections into `data.qpdata`. Skipped entirely when
`skip_detection` is True.

In [3]:
STAGES = ["run_braian_detection.groovy", "02_detect_classify.groovy",
          "03_export_region_table.groovy"]

if PARAMS["skip_detection"]:
    print("skip_detection=True — using the exports already in results/.")
    print("Commands this WOULD run:")
    for script in STAGES:
        print("  " + cc.shell_quote(cc.qupath_command(PROJECT, script)))
else:
    for script in STAGES:
        cmd = cc.qupath_command(PROJECT, script)
        rc = cc.run_qupath(cmd, dry_run=PARAMS["dry_run"],
                           log_path=RESULTS / f"batch__{Path(script).stem}.log")
        if rc > 0:
            raise RuntimeError(f"{script} failed with exit {rc}")
    slices = cc.find_slices(PROJECT)  # re-scan: the export may have added slices

skip_detection=True — using the exports already in results/.
Commands this WOULD run:
  /home/jflab/section-pipeline/tools/QuPath/bin/QuPath script -p '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/project.qpproj' -s '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/scripts/run_braian_detection.groovy'
  /home/jflab/section-pipeline/tools/QuPath/bin/QuPath script -p '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/project.qpproj' -s '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/scripts/02_detect_classify.groovy'
  /home/jflab/section-pipeline/tools/QuPath/bin/QuPath script -p '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/project.qpproj' -s '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/scripts/03_export_region_table.groovy'


## 3. Per-slice QC summary &nbsp;<sub>CHEAP</sub>

One PASS/FLAG row per slice. `overall` is decided by the blocking gates only; advisory
gates report their number without voting.

In [4]:
qc = cc.gate_table(PROJECT, TH)
display(cc.summary_view(qc))

qc.to_csv(RESULTS / PARAMS["qc_csv"], index=False)
print(f"QC table -> {RESULTS / PARAMS['qc_csv']}")

n_flag = int((qc["overall"] == cc.FLAG).sum())
print(f"\n{len(qc) - n_flag}/{len(qc)} slice(s) PASS, {n_flag} FLAG")
if n_flag:
    for _, r in qc[qc["overall"] == cc.FLAG].iterrows():
        print(f"  {r['slice']:<24} {r['flagged']}")
    print("\nA FLAG is a prompt to look, not an automatic stop: check whether the number is")
    print("wrong (retune in 01_calibrate) or the BAND is wrong for this prep (widen it in")
    print("PARAMS['thresholds'] and write down why).")

,slice,nucleus_area_peak_um2,k_swing_pp,white_matter_density,ventricle_density,grey_density_median,total_density,overall
0,wBA1-3_s1,27.5 (FLAG),1.5 (PASS),"4,774.2 (FLAG)","1,317.8 (FLAG)","4,048.0 (FLAG)","4,106.4 (PASS)",FLAG
1,wBA1-3_s2,32.5 (FLAG),1.2 (PASS),"4,501.9 (FLAG)","1,202.9 (FLAG)","4,057.2 (FLAG)","4,013.9 (PASS)",FLAG
2,wBA1-3_s3,32.5 (FLAG),1.2 (PASS),"4,788.0 (FLAG)","2,265.5 (FLAG)","4,008.0 (FLAG)","3,982.5 (PASS)",FLAG
3,wBA1-3_s4,27.5 (FLAG),0.8 (PASS),"4,735.2 (FLAG)","1,259.0 (FLAG)","4,125.4 (FLAG)","4,217.7 (PASS)",FLAG
4,wBA1-3_s5,27.5 (FLAG),0.8 (PASS),"4,996.5 (FLAG)","1,424.0 (FLAG)","3,950.4 (FLAG)","4,101.6 (PASS)",FLAG


QC table -> /home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/results/cockpit_qc_summary.csv

0/5 slice(s) PASS, 5 FLAG
  wBA1-3_s1                nucleus_area_peak_um2, white_matter_density, ventricle_density
  wBA1-3_s2                nucleus_area_peak_um2, white_matter_density, ventricle_density
  wBA1-3_s3                nucleus_area_peak_um2, white_matter_density, ventricle_density
  wBA1-3_s4                nucleus_area_peak_um2, white_matter_density, ventricle_density
  wBA1-3_s5                nucleus_area_peak_um2, white_matter_density, ventricle_density

A FLAG is a prompt to look, not an automatic stop: check whether the number is
wrong (retune in 01_calibrate) or the BAND is wrong for this prep (widen it in
PARAMS['thresholds'] and write down why).


In [5]:
# Per-slice view of the key gate: white matter must sit BELOW cortex.
rows = []
for s in slices:
    if s.regions_tsv is None:
        continue
    reg = cc.load_regions_tsv(s.regions_tsv, anchor_ch)
    cortex = float(np.nanmean([cc.region_density(reg, a)[2] for a in TH.cortex_acronyms]))
    for acr in TH.cortex_acronyms + TH.white_matter_acronyms + TH.ventricle_acronyms:
        d = cc.region_density(reg, acr)[2]
        if np.isfinite(d):
            rows.append({"slice": s.label, "region": acr, "density": d, "cortex": cortex})

dens = pd.DataFrame(rows)
if not dens.empty:
    piv = dens.pivot_table(index="slice", columns="region", values="density")
    order = [r for r in TH.cortex_acronyms + TH.white_matter_acronyms + TH.ventricle_acronyms
             if r in piv.columns]
    piv = piv[order]
    ax = piv.plot(kind="bar", figsize=(11, 4), width=0.8)
    limit = dens.groupby("slice")["cortex"].first().mean() * TH.white_matter_ratio_max
    ax.axhline(limit, color="#C44E52", ls="--", lw=1.4,
               label=f"white-matter limit ({TH.white_matter_ratio_max:g}× cortex)")
    ax.set(ylabel=f"{config.anchor_name} density (per mm²)", xlabel="",
           title="Detection-quality check — cortex vs white matter vs ventricle")
    ax.legend(frameon=False, ncol=3, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("no regions.tsv exports — density comparison skipped")

/tmp/ipykernel_3109631/4268648935.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Classification & k-sweep &nbsp;<sub>CHEAP</sub>

Positive calls use the robust self-calibrating cut imported from `k_sweep_readout`:
`threshold = median + k · 1.4826 · MAD`, derived per marker on that marker's own
classifiable population (`class != Excluded`, finite `<marker>_bgsub`).

Counts and the conditional are driven by whichever markers `pipeline.yml` declares —
a single-marker project simply produces no `Double+` and no conditional.

In [6]:
import k_sweep_readout as ksr

counts, sweep = [], []
for s in slices:
    if s.percell_tsv is None:
        continue
    df = cc.load_percell_for_slice(s)
    tokens = cc.resolve_marker_tokens(df, config)
    if not tokens:
        continue

    row = {"slice": s.label, f"{config.anchor_name}_total": len(df),
           "excluded": int((df["class"] == "Excluded").sum())}
    pos = {}
    for name, tok in tokens.items():
        col = f"{tok}_bgsub"
        mask = ksr.classifiable_mask(df, tok)
        thr = ksr.robust_threshold(df.loc[mask, col].to_numpy(float), PARAMS["k_report"])
        pos[name] = mask & (df[col] >= thr)
        row[f"{name}+"] = int(pos[name].sum())
    if config.emit_double and len(pos) >= 2:
        both = np.logical_and.reduce([v.to_numpy() for v in pos.values()])
        row["Double+"] = int(both.sum())
    counts.append(row)

    names = list(tokens)
    if len(names) >= 2:
        tgt = PARAMS["target_marker"] or names[0]
        cnd = PARAMS["condition_marker"] or names[1]
        for k in PARAMS["k_values"]:
            rate, numer, denom = cc.conditional_positive_rate(
                df, tokens[tgt], tokens[cnd], k)
            sweep.append({"slice": s.label, "k": k, "target": tgt, "condition": cnd,
                          "rate": rate, "numer": numer, "denom": denom})

counts = pd.DataFrame(counts)
sweep = pd.DataFrame(sweep)
print(f"Counts at k={PARAMS['k_report']:g}")
display(counts)

Counts at k=3


,slice,DAPI_total,excluded,Fos+,TdT+,Double+
0,wBA1-3_s1,195823,2368,16294,13232,5384
1,wBA1-3_s2,195401,2471,15698,13283,5619
2,wBA1-3_s3,202692,3251,18962,14695,6599
3,wBA1-3_s4,199171,2354,16013,12539,5179
4,wBA1-3_s5,206624,2652,16654,14322,6143


In [7]:
if sweep.empty:
    print(f"Single marker ({', '.join(MARKERS)}) — no conditional to sweep. "
          "This is the expected single-marker path, not an error.")
else:
    tgt, cnd = sweep["target"].iloc[0], sweep["condition"].iloc[0]
    piv = sweep.pivot_table(index="slice", columns="k", values="rate") * 100
    piv.columns = [f"k={c:g}" for c in piv.columns]
    piv["swing_pp"] = piv.max(axis=1) - piv.min(axis=1)
    piv["stable"] = np.where(piv["swing_pp"] <= TH.k_swing_max_pp, "PASS", "FLAG")
    print(f"P({tgt}+ | {cnd}+) across k  —  FLAG if swing > {TH.k_swing_max_pp:g} pp")
    display(piv.style.format({c: "{:.1f}%" for c in piv.columns if c.startswith("k=")}
                            | {"swing_pp": "{:.1f}"}))

    fig, ax = plt.subplots(figsize=(8, 4))
    for lbl, g in sweep.groupby("slice"):
        ax.plot(g["k"], g["rate"] * 100, marker="o", label=lbl)
    ax.axvline(PARAMS["k_report"], color="#333", ls=":", lw=1,
               label=f"reported k={PARAMS['k_report']:g}")
    ax.set(xlabel="robust multiplier k", ylabel=f"P({tgt}+ | {cnd}+)  (%)",
           title="Stability of the reactivation readout across the classification cut")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

P(Fos+ | TdT+) across k  —  FLAG if swing > 10 pp


,k=2,k=2.5,k=3,swing_pp,stable
slice,,,,,
wBA1-3_s1,42.2%,41.9%,40.7%,1.5,PASS
wBA1-3_s2,43.5%,43.3%,42.3%,1.2,PASS
wBA1-3_s3,46.1%,45.7%,44.9%,1.2,PASS
wBA1-3_s4,41.7%,42.1%,41.3%,0.8,PASS
wBA1-3_s5,43.6%,43.7%,42.9%,0.8,PASS


/tmp/ipykernel_3109631/1967819486.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Region-of-interest CSV &nbsp;<sub>CHEAP</sub>

Delegates to `cockpit_regions` (increment 2) for the exclusion-aware hierarchy roll-up
and hemisphere split — not rebuilt here. It consumes `*__region_table.tsv`; projects
exported before that table was consolidated need a re-export first.

In [8]:
status = cc.region_table_status(PROJECT)
regions_df = None

if not status["ready"]:
    print("Region-of-interest CSV skipped.\n  " + status["hint"])
else:
    wanted = PARAMS["regions"] or creg.read_regions_file(PROJECT)
    print(f"regions: {wanted if wanted else '(all regions — no regions_of_interest.txt)'}")
    regions_df = creg.build_readout(PROJECT, regions=wanted)
    out = creg.write_csv(regions_df, RESULTS / PARAMS["regions_csv"])
    print(f"{len(regions_df):,} rows -> {out}")
    display(regions_df.head(12))

Region-of-interest CSV skipped.
  No *__region_table.tsv found (only the older *__region_area.tsv, which has areas but no counts). Re-export with 03_export_region_table.groovy: /home/jflab/section-pipeline/tools/QuPath/bin/QuPath script -p '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/project.qpproj' -s '/home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/scripts/03_export_region_table.groovy'


## 6. Readout & plots &nbsp;<sub>CHEAP</sub>

The primary readout is marker-centric: **reactivation** P(target+ | condition+) and
**target density** per region.

> Reactivation is reported as **enrichment over the section's own baseline** —
> `P(Fos+|TdT+) / P(Fos+|anchor)` — not as a raw ratio. A raw ratio inflates regions that
> are simply Fos-dense: a region can read 69% raw yet be only 1.4× enriched, while another
> at a lower raw rate is genuinely ~4× enriched. Both columns are emitted so the raw number
> stays visible.

In [9]:
K = PARAMS["k_report"]
wanted = PARAMS["regions"] or creg.read_regions_file(PROJECT)

readout = []
for s in slices:
    if s.percell_tsv is None:
        continue
    df = cc.load_percell_for_slice(s)
    tokens = cc.resolve_marker_tokens(df, config)
    if not tokens:
        continue
    names = list(tokens)
    tgt = PARAMS["target_marker"] or names[0]
    cnd = PARAMS["condition_marker"] or (names[1] if len(names) > 1 else None)

    thr = {n: ksr.robust_threshold(
               df.loc[ksr.classifiable_mask(df, t), f"{t}_bgsub"].to_numpy(float), K)
           for n, t in tokens.items()}
    regions_here = wanted or sorted(df["acronym"].dropna().unique())

    # Section-wide baseline P(target+ | anchor) for the enrichment denominator.
    tmask_all = ksr.classifiable_mask(df, tokens[tgt])
    base = float((df.loc[tmask_all, f"{tokens[tgt]}_bgsub"] >= thr[tgt]).mean()) \
        if tmask_all.any() else np.nan

    for acr in regions_here:
        sub = df[df["acronym"] == acr]
        if sub.empty:
            continue
        tm = ksr.classifiable_mask(sub, tokens[tgt])
        tpos = tm & (sub[f"{tokens[tgt]}_bgsub"] >= thr[tgt])
        row = {"slice": s.label, "region": acr, "k": K,
               f"{config.anchor_name}_n": len(sub),
               f"{tgt}+_n": int(tpos.sum()),
               f"{tgt}+_frac": float(tpos.sum() / tm.sum()) if tm.sum() else np.nan,
               f"{tgt}+_baseline_frac": base}
        if cnd is not None:
            cm = ksr.classifiable_mask(sub, tokens[cnd])
            cpos = cm & (sub[f"{tokens[cnd]}_bgsub"] >= thr[cnd])
            both = tm & cm
            denom = int((cpos & both).sum())
            numer = int((cpos & tpos & both).sum())
            raw = numer / denom if denom else np.nan
            row.update({f"{cnd}+_n": int(cpos.sum()), "Double+_n": numer,
                        "reactivation_raw": raw,
                        "reactivation_enrichment": raw / base if base else np.nan})
        readout.append(row)

readout = pd.DataFrame(readout)
if readout.empty:
    print("no per-cell exports — readout skipped")
else:
    readout.to_csv(RESULTS / PARAMS["readout_csv"], index=False)
    print(f"{len(readout):,} rows -> {RESULTS / PARAMS['readout_csv']}")
    display(readout.head(12))

866 rows -> /home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/results/cockpit_readout.csv


,slice,region,k,DAPI_n,Fos+_n,Fos+_frac,Fos+_baseline_frac,TdT+_n,Double+_n,reactivation_raw,reactivation_enrichment
0,wBA1-3_s1,(no region),3.0,7,1,0.142857,0.084226,1,1,1.000000,11.872775
1,wBA1-3_s1,AAA,3.0,1697,82,0.048321,0.084226,97,35,0.360825,4.283991
2,wBA1-3_s1,AIp1,3.0,379,55,0.145119,0.084226,33,20,0.606061,7.195621
3,wBA1-3_s1,AIp2/3,3.0,1130,63,0.055752,0.084226,30,8,0.266667,3.166073
4,wBA1-3_s1,AIp5,3.0,991,85,0.085772,0.084226,30,15,0.500000,5.936388
5,wBA1-3_s1,AIp6a,3.0,756,111,0.146825,0.084226,74,31,0.418919,4.973730
6,wBA1-3_s1,AIp6b,3.0,19,0,0.000000,0.084226,0,0,NaN,NaN
7,wBA1-3_s1,AMd,3.0,1023,50,0.048876,0.084226,94,17,0.180851,2.147204
8,wBA1-3_s1,AMv,3.0,725,41,0.056552,0.084226,53,12,0.226415,2.688176
9,wBA1-3_s1,AV,3.0,634,23,0.036278,0.084226,96,16,0.166667,1.978796


In [10]:
if readout.empty:
    print("nothing to plot")
else:
    tgt = PARAMS["target_marker"] or list(cc.resolve_marker_tokens(
        cc.load_percell_for_slice(next(s for s in slices if s.percell_tsv)), config))[0]

    # Rank regions by evidence so the plot is not dominated by 3-cell regions.
    agg = (readout.groupby("region")
           .agg(**{f"{tgt}+_frac": (f"{tgt}+_frac", "mean"),
                   "n": (f"{config.anchor_name}_n", "sum")})
           .query("n >= 200").sort_values(f"{tgt}+_frac", ascending=False))
    top = agg.head(15)

    has_enrich = ("reactivation_enrichment" in readout.columns
                  and readout["reactivation_enrichment"].notna().any())
    fig, axes = plt.subplots(1, 2 if has_enrich else 1,
                             figsize=(13 if has_enrich else 7, 5), squeeze=False)

    ax = axes[0][0]
    ax.barh(top.index, top[f"{tgt}+_frac"] * 100, color="#4C72B0")
    ax.set(xlabel=f"{tgt}+ fraction of {config.anchor_name} (%)",
           title=f"{tgt}+ rate by region (k={PARAMS['k_report']:g}, ≥200 cells)")
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)

    if has_enrich:
        enr = (readout.groupby("region")
               .agg(enrichment=("reactivation_enrichment", "mean"),
                    n=(f"{config.anchor_name}_n", "sum"))
               .query("n >= 200").dropna()
               .sort_values("enrichment", ascending=False).head(15))
        ax2 = axes[0][1]
        ax2.barh(enr.index, enr["enrichment"], color="#55A868")
        ax2.axvline(1.0, color="#C44E52", ls="--", lw=1.4, label="no enrichment (1×)")
        ax2.set(xlabel="reactivation enrichment (× section baseline)",
                title="Reactivation enrichment by region")
        ax2.invert_yaxis()
        ax2.legend(frameon=False)
        ax2.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()

/tmp/ipykernel_3109631/3495936140.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
print("=" * 72)
print(f"SERIES SUMMARY — {PROJECT.name}")
print("=" * 72)
print(f"markers        : {', '.join(MARKERS)}   Double+: {'yes' if config.emit_double else 'no'}")
print(f"slices         : {len(slices)}")
if not qc.empty:
    print(f"QC             : {int((qc['overall'] == cc.PASS).sum())}/{len(qc)} PASS")
if not counts.empty:
    tot = counts.drop(columns=["slice"]).sum()
    print(f"totals (k={PARAMS['k_report']:g}) : "
          + "  ".join(f"{k}={int(v):,}" for k, v in tot.items()))
if not sweep.empty:
    at_k = sweep[np.isclose(sweep["k"], PARAMS["k_report"])]
    tgt, cnd = at_k["target"].iloc[0], at_k["condition"].iloc[0]
    print(f"P({tgt}+|{cnd}+)  : {at_k['rate'].mean()*100:.1f}% "
          f"(range {at_k['rate'].min()*100:.1f}–{at_k['rate'].max()*100:.1f}% across slices)")
print("\noutputs:")
for f in (PARAMS["qc_csv"], PARAMS["readout_csv"]) + ((PARAMS["regions_csv"],) if regions_df is not None else ()):
    print(f"  {RESULTS / f}")
print("\nNOTE: these are per-SECTION numbers. Aggregate to the animal level before any")
print("group comparison — sections are not independent (CLAUDE.md stats convention).")

SERIES SUMMARY — wBA_1-3_2-1_072026
markers        : Fos, TdT   Double+: yes
slices         : 5
QC             : 0/5 PASS
totals (k=3) : DAPI_total=999,711  excluded=13,096  Fos+=83,621  TdT+=68,071  Double+=28,924
P(Fos+|TdT+)  : 42.4% (range 40.7–44.9% across slices)

outputs:
  /home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/results/cockpit_qc_summary.csv
  /home/jflab/Analysis/wBA 1-3 2-1 072026/wBA_1-3_2-1_072026/results/cockpit_readout.csv

NOTE: these are per-SECTION numbers. Aggregate to the animal level before any
group comparison — sections are not independent (CLAUDE.md stats convention).
